# CreditIQ — Data Split & Leakage Investigation

This notebook validates the integrity of the frozen Train, Validation, and Test datasets used in the CreditIQ credit-risk modeling pipeline.

The purpose is to ensure:

- No applicant appears in multiple datasets.
- Train, Validation, and Test have identical feature schemas.
- Target distributions are reasonable.
- The split remains fixed throughout the modeling pipeline.
- WOE/IV is learned only from the training dataset.
- Validation and Test data remain unseen during WOE/IV fitting.

In [1]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("..")

DATA_DIR = PROJECT_ROOT / "data" / "processed"

train = pd.read_parquet(DATA_DIR / "train.parquet")
validation = pd.read_parquet(DATA_DIR / "validation.parquet")
test = pd.read_parquet(DATA_DIR / "test.parquet")

print("TRAIN:", train.shape)
print("VALIDATION:", validation.shape)
print("TEST:", test.shape)

TRAIN: (215257, 186)
VALIDATION: (46127, 186)
TEST: (46127, 186)


## 1. Dataset Split

The engineered applicant dataset contains 307,511 applicants.

The dataset is separated into three fixed subsets:

- Training: 215,257 applicants
- Validation: 46,127 applicants
- Test: 46,127 applicants

The split is preserved throughout the project and is not recreated during WOE/IV processing or model training.

In [2]:
print("TRAIN:", train.shape)
print("VALIDATION:", validation.shape)
print("TEST:", test.shape)

print("\nTarget rates:")
print("Train:", train["TARGET"].mean())
print("Validation:", validation["TARGET"].mean())
print("Test:", test["TARGET"].mean())

TRAIN: (215257, 186)
VALIDATION: (46127, 186)
TEST: (46127, 186)

Target rates:
Train: 0.07346102565770218
Validation: 0.09391462700804301
Test: 0.10145901532724869


## 2. Applicant ID Overlap Check

`SK_ID_CURR` uniquely identifies an applicant.

No applicant should appear in more than one dataset.

In [3]:
train_ids = set(train["SK_ID_CURR"])
validation_ids = set(validation["SK_ID_CURR"])
test_ids = set(test["SK_ID_CURR"])

print("Train-Validation overlap:", len(train_ids & validation_ids))
print("Train-Test overlap:", len(train_ids & test_ids))
print("Validation-Test overlap:", len(validation_ids & test_ids))

Train-Validation overlap: 0
Train-Test overlap: 0
Validation-Test overlap: 0


## 3. Schema Consistency

All three datasets must contain the same columns so that the same preprocessing and modeling pipeline can be applied consistently.

In [4]:
print(
    "Train = Validation columns:",
    list(train.columns) == list(validation.columns)
)

print(
    "Train = Test columns:",
    list(train.columns) == list(test.columns)
)

Train = Validation columns: True
Train = Test columns: True


## 4. WOE/IV Leakage Prevention

WOE and Information Value use the target variable and therefore must be learned using the training data only.

CreditIQ follows:

Train → calculate WOE/IV → select features

Validation → apply the TRAIN WOE mappings

Test → apply the TRAIN WOE mappings

Validation and Test are never used to calculate WOE/IV or select features.

## 5. Conclusion

The Train, Validation, and Test datasets are correctly segregated with zero applicant overlap and consistent schemas.

The split is treated as frozen for the remainder of the project.

The WOE/IV pipeline uses only training information to learn transformations and feature selection, preventing target leakage from Validation or Test data.